# XAI stability -- TESTS

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [2]:
import time
import numpy as np
import pandas as pd
import nbimporter

# Utils
import torch
import os
import pickle
from sklearn.base import clone

import xgboost as xgb
from sklearn.neural_network import MLPClassifier


import Taylor_Explainer as texp
import XAI_stability_metrics as stab


import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
import openxai

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

# Perturbation definition

In [4]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.05
perturbation_flip_percentage= 0.01
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Loading data

# 1. Synthetic - 20 features - Numeric

In [5]:
ox_path= 'data/synth_OX_20/processed/'

train_ox= pd.read_csv(ox_path + 'X_train.csv')
test_ox = pd.read_csv(ox_path + 'X_test.csv')
labels_train_ox= pd.read_csv(ox_path + 'y_train.csv')
labels_test_ox = pd.read_csv(ox_path + 'y_test.csv')

In [6]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn1_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn1_model_ox.predict(test_ox))
acc_nn1_ox

0.83

In [7]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ox= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn2_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn2_model_ox.predict(test_ox))
acc_nn2_ox

0.83

In [8]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.84

In [9]:
# definitions

# get n and m parameters from train and labels_train
n_ox, m_ox= texp.get_n_m_sizes(train_ox, labels_train_ox)

# conversion of train_ox and labels_train_ox data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ox.values)
tn_lb_tr= torch.from_numpy(labels_train_ox.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ox= dict()

h_min_dist_ox= texp.get_minimum_distance(train_ox)

# T-Exp explanation settings
descriptor_ox['h_min']= h_min_dist_ox
descriptor_ox['h_max']= 1
descriptor_ox['jacobian_eps']= 1e-3
descriptor_ox['max_itr']= 30
descriptor_ox['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ox['num_samples']= 30
descriptor_ox['num_perts']= 10   # RIS/ROS
descriptor_ox['pert_max_distance']= (h_min_dist_ox/2)
descriptor_ox['num_runs']= 10    # RES
descriptor_ox['feature_metadata']= ['c'] * n_ox
descriptor_ox['p_norm']= 2
descriptor_ox['eps_norm']= 1e-6
descriptor_ox['top_k']= 0
descriptor_ox['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ox['top_k'])

In [12]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
print(stab.relative_stability(nn1_model_ox, test_ox, labels_test_ox, perturbation, 
                              descriptor_ox, is_model_NN=True))

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


RIS/ROS --
Instance 1 out 200


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

{'t_exp_ris_max': 3581833.4059667448, 'std(t_exp_ris_max)': 301475.42113264994, 'shap_ris_max': 191456.55362358343, 'std(shap_ris_max)': 22865.4698432637, 'lime_ris_max': 1672.8280951257923, 'std(lime_ris_max)': 181.5826482502844, 't_exp_ris_mean': 10541.434185318376, 'std(t_exp_ris_mean)': 107348.73119397482, 'shap_ris_mean': 2279.839810817384, 'std(shap_ris_mean)': 7214.962038686275, 'lime_ris_mean': 19.971364323724153, 'std(lime_ris_mean)': 90.07310458383743, 't_exp_ros_max': 1589785839973.9038, 'std(t_exp_ros_max)': 112331279183.78777, 'shap_ros_max': 34038901.65534534, 'std(shap_ros_max)': 3618620.293646344, 'lime_ros_max': 3307263.8191635464, 'std(lime_ros_max)': 420671.55606464954, 't_exp_ros_mean': 2616267098.652799, 'std(t_exp_ros_mean)': 34724101184.236305, 'shap_ros_mean': 258327.66895228557, 'std(shap_ros_mean)': 1232233.0972036268, 'lime_ros_mean': 32557.207287913287, 'std(lime_ros_mean)': 167263.39841528065, 'shap_kernel_ris_max': 2433.2573967926214, 'std(shap_kernel_ris_

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [10]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
print(stab.run_stability(nn1_model_ox, test_ox, labels_test_ox, descriptor_ox, is_model_NN=True))

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 1 out 200


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

{'texp_res': 9.455164053283262e-15, 'shap_res': 0.1590239761796944, 'shap_kernel_res': 0.056604184838021855, 'shap_exact_res': '--', 'lime_res': 5.816542563864309e-17, 'itGd_res': 1.1149867635186251e-14, 'iXGd_res': 8.145812e-06, 'dLif_res': 8.302823e-06, 'lwrp_res': 8.777078e-06, 'smoothG_res': 7.956445320772315, 'vanillaG_res': 1.3662861e-05, 'GuidBprop_res': 1.3662861e-05, 'occlusion_res': 5.2452087e-06}

--- 8911.28 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [11]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
print(stab.relative_stability(nn2_model_ox, test_ox, labels_test_ox, perturbation, 
                              descriptor_ox, is_model_NN=True))

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


RIS/ROS --
Instance 1 out 200


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

{'t_exp_ris_max': 6939.966276039339, 'std(t_exp_ris_max)': 549.9280591060078, 'shap_ris_max': 106394.81636766148, 'std(shap_ris_max)': 12011.547934715905, 'lime_ris_max': 2093.5231336264173, 'std(lime_ris_max)': 149.86502474357, 't_exp_ris_mean': 56.02532870233234, 'std(t_exp_ris_mean)': 359.39813091447814, 'shap_ris_mean': 1119.2732882209987, 'std(shap_ris_mean)': 4062.692266144707, 'lime_ris_mean': 8.531344983936012, 'std(lime_ris_mean)': 74.9714142154444, 't_exp_ros_max': 180597.4688749912, 'std(t_exp_ros_max)': 12781.577077791233, 'shap_ros_max': 2988685.0947643872, 'std(shap_ros_max)': 220056.467106025, 'lime_ros_max': 5250.451241172367, 'std(lime_ros_max)': 539.4057840859969, 't_exp_ros_mean': 197.47315362746383, 'std(t_exp_ros_mean)': 1340.8154289169886, 'shap_ros_mean': 3856.2322195740244, 'std(shap_ros_mean)': 23717.85278906305, 'lime_ros_mean': 20.016176280077534, 'std(lime_ros_mean)': 117.5794513134579, 'shap_kernel_ris_max': 4859.299944036011, 'std(shap_kernel_ris_max)': 34

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [12]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
print(stab.run_stability(nn2_model_ox, test_ox, labels_test_ox, descriptor_ox, is_model_NN=True))

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --


`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


Instance 1 out 200


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

{'texp_res': 4.0943002132167226e-15, 'shap_res': 0.15096620528743804, 'shap_kernel_res': 0.05286108668538363, 'shap_exact_res': '--', 'lime_res': 5.601874435361059e-17, 'itGd_res': 7.838739178637249e-15, 'iXGd_res': 3.1411776e-06, 'dLif_res': 3.1047741e-06, 'lwrp_res': 3.9801726e-06, 'smoothG_res': 4.609237031590465, 'vanillaG_res': 7.0414253e-06, 'GuidBprop_res': 7.0414253e-06, 'occlusion_res': 2.311554e-06}

--- 16773.41 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


In [13]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
print(stab.relative_stability(nn3_model_ox, test_ox, labels_test_ox, perturbation, 
                              descriptor_ox, is_model_NN=True))

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RIS/ROS --


`should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.


Instance 1 out 200


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


{'t_exp_ris_max': 9883.167741490019, 'std(t_exp_ris_max)': 724.1552936390742, 'shap_ris_max': 161023.26459530706, 'std(shap_ris_max)': 18884.08466606947, 'lime_ris_max': 125.23023867181428, 'std(lime_ris_max)': 9.288623104474157, 't_exp_ris_mean': 47.66760338967645, 'std(t_exp_ris_mean)': 190.69174039952586, 'shap_ris_mean': 1685.5574136318628, 'std(shap_ris_mean)': 5682.2843010441975, 'lime_ris_mean': 1.4457846936396919, 'std(lime_ris_mean)': 4.69466973980941, 't_exp_ros_max': 72377.13231677927, 'std(t_exp_ros_max)': 6912.021735332718, 'shap_ros_max': 4487201.568946395, 'std(shap_ros_max)': 318004.48935245024, 'lime_ros_max': 9439.247110636987, 'std(lime_ros_max)': 671.7001945583183, 't_exp_ros_mean': 243.76744908672234, 'std(t_exp_ros_mean)': 1119.6473489245664, 'shap_ros_mean': 5068.45797281265, 'std(shap_ros_mean)': 35630.91825643966, 'lime_ros_mean': 10.875264534668972, 'std(lime_ros_mean)': 68.1580858559859, 'shap_kernel_ris_max': 2874.3006935620547, 'std(shap_kernel_ris_max)': 2

In [10]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
print(stab.run_stability(nn3_model_ox, test_ox, labels_test_ox, descriptor_ox, is_model_NN=True))

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

RES --
Instance 1 out 200


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

{'texp_res': 5.630757419431282e-15, 'shap_res': 0.17422857751305387, 'shap_kernel_res': 0.04951970020924617, 'shap_exact_res': '--', 'lime_res': 5.916685065386561e-17, 'itGd_res': 5.208590938883218e-15, 'iXGd_res': 4.5775437e-06, 'dLif_res': 4.874844e-06, 'lwrp_res': 4.572496e-06, 'smoothG_res': 5.9312568401515, 'vanillaG_res': 7.6779015e-06, 'GuidBprop_res': 7.6779015e-06, 'occlusion_res': 2.5788913e-06}

--- 22487.87 seconds ---


Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


# 2. Adult Income

In [ ]:
ad_path= 'data/adult/processed/'

train_ad= pd.read_csv(ad_path + 'X_train.csv')
test_ad = pd.read_csv(ad_path + 'X_test.csv')
labels_train_ad= pd.read_csv(ad_path + 'y_train.csv')
labels_test_ad = pd.read_csv(ad_path + 'y_test.csv')

# 3. Chess (kr-vs-kp)

In [ ]:
ch_path= 'data/chess/processed/'

train_ch= pd.read_csv(ch_path + 'X_train.csv')
test_ch = pd.read_csv(ch_path + 'X_test.csv')
labels_train_ch= pd.read_csv(ch_path + 'y_train.csv')
labels_test_ch = pd.read_csv(ch_path + 'y_test.csv')

# 4. COMPAS

In [ ]:
cpas_path= 'data/compas/processed/'

train_cpas= pd.read_csv(cpas_path + 'X_train.csv')
test_cpas = pd.read_csv(cpas_path + 'X_test.csv')
labels_train_cpas= pd.read_csv(cpas_path + 'y_train.csv')
labels_test_cpas = pd.read_csv(cpas_path + 'y_test.csv')

# 5. Diabetes

In [ ]:
diab_path= 'data/diabetes/processed/'

train_diab= pd.read_csv(diab_path + 'X_train.csv')
test_diab = pd.read_csv(diab_path + 'X_test.csv')
labels_train_diab= pd.read_csv(diab_path + 'y_train.csv')
labels_test_diab = pd.read_csv(diab_path + 'y_test.csv')

# 6. German Credit

In [ ]:
ger_path= 'data/german/processed/'

train_ger= pd.read_csv(ger_path + 'X_train.csv')
test_ger = pd.read_csv(ger_path + 'X_test.csv')
labels_train_ger= pd.read_csv(ger_path + 'y_train.csv')
labels_test_ger = pd.read_csv(ger_path + 'y_test.csv')

# 7. HELOC

In [ ]:
hel_path= 'data/heloc/processed/'

train_hel= pd.read_csv(hel_path + 'X_train.csv')
test_hel = pd.read_csv(hel_path + 'X_test.csv')
labels_train_hel= pd.read_csv(hel_path + 'y_train.csv')
labels_test_hel = pd.read_csv(hel_path + 'y_test.csv')

# 8. HIGGS

In [ ]:
hig_path= 'data/higgs/processed/'

train_hig= pd.read_csv(hig_path + 'X_train.csv')
test_hig = pd.read_csv(hig_path + 'X_test.csv')
labels_train_hig= pd.read_csv(hig_path + 'y_train.csv')
labels_test_hig = pd.read_csv(hig_path + 'y_test.csv')

# 9. Independent

In [ ]:
indep_path= 'data/independent/processed/'

train_indep= pd.read_csv(indep_path + 'X_train.csv')
test_indep = pd.read_csv(indep_path + 'X_test.csv')
labels_train_indep= pd.read_csv(indep_path + 'y_train.csv')
labels_test_indep = pd.read_csv(indep_path + 'y_test.csv')

# 10. LSA - Law School Admission

In [ ]:
lsa_path= 'data/law_school_admission/processed/'

train_lsa= pd.read_csv(lsa_path + 'X_train.csv')
test_lsa = pd.read_csv(lsa_path + 'X_test.csv')
labels_train_lsa= pd.read_csv(lsa_path + 'y_train.csv')
labels_test_lsa = pd.read_csv(lsa_path + 'y_test.csv')